In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as cp
from scipy.interpolate import griddata
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from scipy.interpolate import interp1d
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [2]:
MBH = 1e6
Rp = 17
a = -0.2
N = 500
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_r = TDECalculator('MAMS1Msun', orbit, MBH, Rp, a, N=N)

In [4]:
sample_r = mass_r.rel_whole_star_sample()

In [5]:
radii_r = sample_r['rr']

rtde = mass_r.R_TDE
Lz = mass_r.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_r <= 0.5,
    rtde - radii_r * mass_r.Rstar,
    rtde + radii_r * mass_r.Rstar
)

deltaE = mass_r.Rstar / mass_r.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_r['dEnergy_random'] * deltaE 
dLz = mass_r.dLz_random
dQ = mass_r.dQ_random
mass_ratio = mass_r.mass_ratio

In [6]:
dT_r = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, Q, dE, dLz, dQ, N)
np.save('dT_r.npy', dT_r.dTs)
gc.collect()

Computing radial periods for 101,880,000 particles ...
  E  range: [0.998692, 1.001308]
  Q  range: [-4.767e+01, 4.767e+01]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 10163/10163 (100%)
  Bound: 50,814,391 / 101,880,000
  Valid roots: 50,814,391
  Valid Lambda_r: 50,814,391
  Quadrature: 1017 chunks ...
    chunk 1017/1017  (100%)
  Successful T_r: 47,656,031 / 101,880,000


20

In [7]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample['dEnergy_random']
    dT_rand = dT / delta
    dMass   = whole_star_sample['dMass']

    bins_E = np.linspace(-2, 2, 1000)
    bins_T = np.logspace(0, 6, 1000)

    # dE: use all particles with finite energy and mass
    valid_E = np.isfinite(dE_rand)
    hist_E, edges_E = np.histogram(dE_rand[valid_E], bins=bins_E,
                                   weights=dMass[valid_E], density=True)

    # dT: only particles with valid finite period within bin range
    valid_T = (np.isfinite(dT_rand) & np.isfinite(dE_rand)
               & (dT_rand >= 1.0) & (dT_rand <= 1e6))
    hist_T, edges_T = np.histogram(dT_rand[valid_T], bins=bins_T,
                                   weights=dMass[valid_T], density=True)

    return {"x": 0.5*(edges_E[:-1]+edges_E[1:]), "y": hist_E}, \
           {"x": 0.5*(edges_T[:-1]+edges_T[1:]), "y": hist_T}

In [8]:
n_ex = TDECalculator('MAMS1Msun', Rp=Rp, a=0.0, N=N)
DeltaE = n_ex.Rstar / n_ex.Rp**2
DeltaT = 1 / DeltaE**1.5

In [9]:
rel_r_E, rel_r_T = make_plot_dicts(sample_r, dT_r.dTs, DeltaT)

In [10]:
import json

adden = "m1_rp17_a0p2"

with open(f"fallback_curves/rel_r_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_E.items()}, f)
with open(f"fallback_curves/rel_r_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_r_T.items()}, f)

In [14]:
print(f"Q passed to FallbackGen retrograde: {Q}")
print(f"total_Q range retrograde: {dT_r.total_Q.min():.3e}, {dT_r.total_Q.max():.3e}")

Q passed to FallbackGen retrograde: 0
total_Q range retrograde: -4.767e+01, 4.767e+01


In [17]:

print(f"finite dTs retrograde: {np.isfinite(dT_r.dTs).sum()}")

finite dTs retrograde: 47656031


In [20]:
print(f"min Lambda_r retrograde: {np.nanmin(dT_r.Lambda_r):.6f}")
print(f"min avg_Tr retrograde: {np.nanmin(dT_r.avg_Tr_arr):.6f}")

min Lambda_r retrograde: 0.709956
min avg_Tr retrograde: 66473.077940


In [22]:
idx_min_r = np.nanargmin(dT_r.dTs)
print(f"retrograde min period particle - E: {dT_r.total_E.ravel()[idx_min_r]:.6f}, Lz: {dT_r.total_Lz.ravel()[idx_min_r]:.6f}, Q: {dT_r.total_Q.ravel()[idx_min_r]:.6f}")

retrograde min period particle - E: 0.998692, Lz: 6.095781, Q: 47.661941


In [25]:
print(f"DeltaT: {DeltaT:.6f}")
print(f"nanmin dT_p.dTs: {np.nanmin(dT_r.dTs):.6f}")

DeltaT: 14410.513237
nanmin dT_p.dTs: 47194.388243
